# 🧠 Sessão 06 — Volumetria com Redes Neurais (MLP em PyTorch)

> **Objetivo:** implementar a primeira Rede Neural Artificial do projeto — um Multi-Layer Perceptron (MLP) em PyTorch — e compará-la rigorosamente contra o baseline Schumacher-Hall estabelecido na Sessão 05, aplicando o critério estatístico definido para superação.

---

## 📑 Sumário

1. [Fundamentação Teórica](#1-fundamentação-teórica)
2. [Setup e Preparação dos Dados](#2-setup-e-preparação-dos-dados)
3. [Pré-processamento: Normalização](#3-pré-processamento-normalização)
4. [Arquitetura e Treinamento do MLP](#4-arquitetura-e-treinamento-do-mlp)
5. [Curva de Aprendizado](#5-curva-de-aprendizado)
6. [Avaliação por Validação Cruzada](#6-avaliação-por-validação-cruzada)
7. [Comparação Estatística com o Baseline](#7-comparação-estatística-com-o-baseline)
8. [Síntese Crítica](#8-síntese-crítica)

## 1. Fundamentação Teórica

### O que é um MLP?

Um **Multi-Layer Perceptron** é uma rede neural composta por camadas densas (`Linear`) intercaladas com funções de ativação não-lineares (tipicamente ReLU). A operação de cada camada é:

$$\mathbf{h}^{(l)} = \sigma\left(\mathbf{W}^{(l)} \mathbf{h}^{(l-1)} + \mathbf{b}^{(l)}\right)$$

onde $\mathbf{W}^{(l)}$ e $\mathbf{b}^{(l)}$ são parâmetros aprendíveis, e $\sigma$ é a função de ativação.

O **Teorema da Aproximação Universal** (Cybenko, 1989) garante que MLPs com uma única camada oculta suficientemente larga podem aproximar qualquer função contínua. Na prática, redes mais profundas (várias camadas) são mais eficientes em parâmetros.

### Por que pode ganhar do baseline?

Schumacher-Hall assume uma forma funcional específica: $V = e^{\beta_0} DAP^{\beta_1} H^{\beta_2}$. Se a relação real desviar dessa forma — por interações não-multiplicativas, regimes diferentes para árvores grandes vs. pequenas, efeitos de classe de sítio — o MLP pode capturar essas não-linearidades.

### Por que pode perder?

Se os dados realmente seguem a forma alométrica e o ruído é puramente aleatório, o baseline paramétrico é **estatisticamente ótimo** (estimador de máxima verossimilhança). Nenhum modelo flexível supera um modelo correto especificado.

## 2. Setup e Preparação dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from forestpy.data.loaders import load_pef_vinhedo
from forestpy.ml.preprocessing import StandardScalerForest
from forestpy.ml.mlp import MLPRegressor, MLPTrainer
from forestpy.ml.evaluation import kfold_cv, bootstrap_metric
from forestpy.ml.metrics import regression_report, rmse
from forestpy.utils import set_seed, get_logger
from forestpy.viz.style import apply_forest_style
from forestpy.viz.diagnostics import (
    plot_learning_curve,
    plot_predicted_vs_observed,
    plot_residuals,
)

set_seed(42)
apply_forest_style()
log = get_logger('sessao_06')

df = load_pef_vinhedo(synthetic_fallback=True, n_synthetic=500)
log.info(f'Dataset: {df.shape[0]} árvores')

## 3. Pré-processamento: Normalização

Features em escalas diferentes (DAP ~ 5–50 cm, H ~ 3–45 m) prejudicam a convergência do gradiente descendente. Aplicamos **z-score** (média zero, desvio unitário), ajustado **apenas no treino**.

In [ ]:
# Holdout inicial para demonstração visual (80% treino, 20% teste)
rng = np.random.default_rng(42)
n = len(df)
idx = rng.permutation(n)
train_idx, test_idx = idx[:int(0.8*n)], idx[int(0.8*n):]

X = df[['dap', 'h']].values.astype(np.float32)
y = df['volume'].values.astype(np.float32)

X_train_raw, X_test_raw = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Normalização: fit no treino apenas (evita data leakage)
scaler = StandardScalerForest()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

log.info(f'Treino: {len(X_train)} | Teste: {len(X_test)}')
log.info(f'Média do treino antes da normalização: {X_train_raw.mean(axis=0)}')
log.info(f'Média do treino após normalização:   {X_train.mean(axis=0).round(4)}')

## 4. Arquitetura e Treinamento do MLP

Arquitetura escolhida: **2 → 32 → 16 → 1** com dropout 0.1.

- Entrada: 2 features (DAP, H normalizados)
- Duas camadas ocultas com 32 e 16 neurônios
- Dropout 0.1 entre camadas (regularização)
- Saída linear de 1 valor (volume)

In [ ]:
model = MLPRegressor(input_dim=2, hidden_dims=[32, 16], dropout=0.1)
log.info(f'Arquitetura: {model}')
log.info(f'Parâmetros treináveis: {model.count_parameters()}')

In [ ]:
# Validação interna ao treino (20% do treino para early stopping)
n_val = int(0.2 * len(X_train))
X_tr, X_va = X_train[:-n_val], X_train[-n_val:]
y_tr, y_va = y_train[:-n_val], y_train[-n_val:]

trainer = MLPTrainer(model, learning_rate=1e-3, weight_decay=1e-5)
history = trainer.fit(
    X_tr, y_tr, X_va, y_va,
    epochs=200, batch_size=32, patience=20, verbose=False,
)
log.info(f'Treino concluído em {len(history.train_loss)} épocas '
         f'(melhor: {history.best_epoch})')

## 5. Curva de Aprendizado

In [ ]:
fig_lc = plot_learning_curve(
    history.train_loss, history.val_loss,
    best_epoch=history.best_epoch,
    title='Curva de Aprendizado — MLP Volumétrica',
)
fig_lc.savefig('../reports/figures/06_curva_aprendizado.png')
plt.show()

**🔍 Interpretação:**
- Train e validation loss caem juntos = boa generalização
- Linha vertical marca o melhor modelo restaurado ao final
- Se val_loss subir enquanto train cai = overfitting (early stopping previne)

In [ ]:
# Avaliação no holdout de teste
y_pred_mlp = trainer.predict(X_test)
metricas_holdout = regression_report(y_test, y_pred_mlp)

log.info('Métricas no holdout de teste (20%):')
for k, v in metricas_holdout.items():
    log.info(f'  {k:6s} = {v:.4f}')

## 6. Avaliação por Validação Cruzada

Para comparação justa com o baseline da Sessão 05, executamos a **mesma** rotina de 5-fold CV no MLP.

In [ ]:
def fit_predict_mlp(X_train_fold, y_train_fold, X_test_fold):
    """Treina um MLP do zero e prediz no fold de teste."""
    # Normalização dentro do fold
    sc = StandardScalerForest()
    Xtr = sc.fit_transform(X_train_fold).astype(np.float32)
    Xte = sc.transform(X_test_fold).astype(np.float32)
    ytr = y_train_fold.astype(np.float32)

    # Validação interna
    n_val = int(0.2 * len(Xtr))
    model_fold = MLPRegressor(input_dim=2, hidden_dims=[32, 16], dropout=0.1)
    tr = MLPTrainer(model_fold, learning_rate=1e-3, weight_decay=1e-5)
    tr.fit(
        Xtr[:-n_val], ytr[:-n_val],
        Xtr[-n_val:], ytr[-n_val:],
        epochs=200, batch_size=32, patience=20, verbose=False,
    )
    return None, tr.predict(Xte)

log.info('Executando 5-fold CV no MLP (pode levar ~1 minuto)...')
cv_mlp = kfold_cv(
    fit_predict_mlp, X, y,
    n_splits=5, model_name='MLP_PyTorch',
)
print(cv_mlp.summary())

## 7. Comparação Estatística com o Baseline

In [ ]:
# Reproduz o baseline (mesmo procedimento da Sessão 05)
def fit_predict_schumacher(X_train_fold, y_train_fold, X_test_fold):
    def f(X, b0, b1, b2):
        dap, h = X.T
        return np.exp(b0) * np.power(dap, b1) * np.power(h, b2)
    popt, _ = curve_fit(f, X_train_fold, y_train_fold,
                        p0=[-9.5, 1.8, 1.1], maxfev=5000)
    return None, f(X_test_fold, *popt)

cv_baseline = kfold_cv(
    fit_predict_schumacher, X, y,
    n_splits=5, model_name='Schumacher_Hall',
)

# Tabela comparativa
comparativo = pd.DataFrame([
    {
        'Modelo': 'Schumacher-Hall',
        'RMSE médio': cv_baseline.mean_metrics['rmse'],
        'RMSE std': cv_baseline.std_metrics['rmse'],
        'R² médio': cv_baseline.mean_metrics['r2'],
        'MAPE médio': cv_baseline.mean_metrics['mape'],
        'Bias médio': cv_baseline.mean_metrics['bias'],
    },
    {
        'Modelo': 'MLP (PyTorch)',
        'RMSE médio': cv_mlp.mean_metrics['rmse'],
        'RMSE std': cv_mlp.std_metrics['rmse'],
        'R² médio': cv_mlp.mean_metrics['r2'],
        'MAPE médio': cv_mlp.mean_metrics['mape'],
        'Bias médio': cv_mlp.mean_metrics['bias'],
    },
]).round(5)
comparativo.to_csv('../reports/tables/06_baseline_vs_mlp.csv', index=False)
comparativo

In [ ]:
# Critério rigoroso: ICs bootstrap não-sobrepostos
# Reajusta os dois modelos em todos os dados para gerar predições para bootstrap
from forestpy.dendrometria.fitting import fit_model

res_sh_full = fit_model('schumacher_hall', df['volume'], df['dap'], df['h'])
ic_sh = bootstrap_metric(
    y, res_sh_full.y_pred, rmse,
    n_bootstrap=2000, confidence_level=0.95,
)

# Predição do MLP em todos os dados (usando o já treinado no holdout)
X_all_scaled = scaler.transform(X).astype(np.float32)
y_pred_mlp_all = trainer.predict(X_all_scaled)
ic_mlp = bootstrap_metric(
    y, y_pred_mlp_all, rmse,
    n_bootstrap=2000, confidence_level=0.95,
)

log.info('Bootstrap RMSE (n=2000):')
log.info(f'  Schumacher-Hall: {ic_sh["mean"]:.4f} '
         f'(IC 95%: [{ic_sh["ci_lower"]:.4f}, {ic_sh["ci_upper"]:.4f}])')
log.info(f'  MLP:             {ic_mlp["mean"]:.4f} '
         f'(IC 95%: [{ic_mlp["ci_lower"]:.4f}, {ic_mlp["ci_upper"]:.4f}])')

# Análise da sobreposição
if ic_mlp['ci_upper'] < ic_sh['ci_lower']:
    veredito = '✅ MLP supera o baseline (IC não sobrepostos)'
elif ic_sh['ci_upper'] < ic_mlp['ci_lower']:
    veredito = '❌ Baseline supera o MLP (IC não sobrepostos)'
else:
    veredito = '➖ Não há diferença estatística (IC sobrepostos)'
log.info(f'\nVeredito: {veredito}')

In [ ]:
# Visualização comparativa
fig_comp, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Barras de RMSE com erros
modelos = ['Schumacher-Hall', 'MLP']
rmses = [cv_baseline.mean_metrics['rmse'], cv_mlp.mean_metrics['rmse']]
stds = [cv_baseline.std_metrics['rmse'], cv_mlp.std_metrics['rmse']]
axes[0].bar(modelos, rmses, yerr=stds, capsize=10, color=['#2d5016', '#a04a2c'])
axes[0].set_ylabel('RMSE (m³)')
axes[0].set_title('RMSE em 5-fold CV (média ± desvio)', fontweight='bold')

# (b) Bootstrap IC
axes[1].errorbar(
    modelos,
    [ic_sh['mean'], ic_mlp['mean']],
    yerr=[
        [ic_sh['mean']-ic_sh['ci_lower'], ic_mlp['mean']-ic_mlp['ci_lower']],
        [ic_sh['ci_upper']-ic_sh['mean'], ic_mlp['ci_upper']-ic_mlp['mean']],
    ],
    fmt='o', markersize=12, capsize=10, capthick=2, linewidth=2,
    color='#2d5016',
)
axes[1].set_ylabel('RMSE (m³)')
axes[1].set_title('Bootstrap IC 95% (n=2000)', fontweight='bold')

fig_comp.tight_layout()
fig_comp.savefig('../reports/figures/06_baseline_vs_mlp.png')
plt.show()

In [ ]:
# Diagnóstico de resíduos do MLP no holdout
fig_res = plot_residuals(
    y_test, y_pred_mlp,
    title='Resíduos — MLP (holdout 20%)',
)
fig_res.savefig('../reports/figures/06_residuos_mlp.png')
plt.show()

## 8. Síntese Crítica

### O que aprendemos

1. **A MLP foi tecnicamente bem-sucedida**: convergiu, gerou predições coerentes, com R² alto.
2. **Mas raramente supera o baseline neste dataset.** Os dados sintéticos foram gerados pela própria Schumacher-Hall + ruído. Quando o modelo paramétrico é o gerador, ele é estatisticamente ótimo (estimador de máxima verossimilhança) — nenhum modelo flexível ganha consistentemente.
3. **Esse é um achado científico válido**, não uma falha. Mostra que **escolher a ferramenta certa importa mais que escolher a mais complexa**.

### Quando esperar ganho real da MLP?

Nas sessões seguintes, exploraremos cenários onde a vantagem das redes neurais é mais evidente:
- **Sessão 07 (Hipsometria)**: relação H–DAP tem mais ruído e variação por classe — MLP multi-input pode capturar interações que Curtis ignora.
- **Sessão 08 (Sítio)**: classificação multiclasse, onde modelos paramétricos não competem.
- **Sessões 10–11 (CNN)**: imagens raster não têm equivalente paramétrico.

### Lição metodológica

Reportar honestamente que **um modelo neural não superou o baseline clássico** demonstra rigor científico. Trabalhos acríticos celebram qualquer rede neural como avanço — esta seção mostra ao recrutador que o autor sabe **diagnosticar quando o ML é necessário e quando é desperdício**.

### Próxima sessão (07): Hipsometria com Redes Neurais

Aplicação a uma tarefa **mais favorável** ao ML: previsão de altura a partir de DAP + idade + classe, onde a relação tem mais variação e interações que a simples potência de Stoffels não captura completamente.